# 0. Full Pipeline

This notebook combines the training and analysis flows for the next-scale AttnRes experiment into a single Colab workflow.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = 'AttnResGPT-next-scale'
REPO_URL = os.environ.get('ATTNRES_REPO_URL', 'https://github.com/your-org/AttnResGPT-next-scale.git')
REPO_DIR = Path('/content') / REPO_NAME
DRIVE_REPO_DIR = Path('/content/drive/MyDrive') / REPO_NAME

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as error:
    print(f'Drive mount skipped: {error}')

if not REPO_DIR.exists():
    if DRIVE_REPO_DIR.exists():
        subprocess.run(['cp', '-r', str(DRIVE_REPO_DIR), str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Using repo at: {REPO_DIR}')


In [ ]:
import torch

print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())


## Stage 1: Required First-Run Experiment

This runs SMALL only, contexts 128 and 512, baseline plus AttnRes, 300 steps each.

In [ ]:
!python experiments/scale_experiment.py --config configs/first_run.yaml --skip-existing

In [ ]:
import pandas as pd
from pathlib import Path

summary_df = pd.read_csv('outputs/logs/run_summaries.csv')
consolidated_df = pd.read_csv('outputs/logs/consolidated_summary_table.csv')
paired_df = pd.read_csv('outputs/logs/paired_comparisons.csv')

display(consolidated_df.sort_values(['size', 'context', 'model']))
display(paired_df.sort_values(['size', 'context']))


## Stage 2: Optional Full Sweep

Only run these after the first-run outputs look healthy.

In [ ]:
# !python experiments/scale_experiment.py --config configs/small.yaml --skip-existing
# !python experiments/scale_experiment.py --config configs/medium.yaml --skip-existing

## Analysis

This section reproduces the analysis notebook inside the same Colab tab.

In [ ]:
import json

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

summary_df = pd.read_csv('outputs/logs/run_summaries.csv')
consolidated_df = pd.read_csv('outputs/logs/consolidated_summary_table.csv')
paired_df = pd.read_csv('outputs/logs/paired_comparisons.csv')

display(consolidated_df.sort_values(['size', 'context', 'model']))
display(paired_df.sort_values(['size', 'context']))


In [ ]:
plot_dir = Path('outputs/plots')
plot_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=summary_df, x='context', y='val_loss', hue='model', style='size', marker='o', ax=axes[0])
axes[0].set_title('Validation Loss vs Context')
sns.lineplot(data=summary_df, x='context', y='perplexity', hue='model', style='size', marker='o', ax=axes[1])
axes[1].set_title('Perplexity vs Context')
fig.tight_layout()
fig.savefig(plot_dir / 'loss_and_perplexity_vs_context.png', dpi=200)
plt.show()


In [ ]:
attnres_rows = summary_df[summary_df['model'] == 'attnres'][['run_name', 'context']].sort_values('context')
for _, row in attnres_rows.iterrows():
    summary_path = Path('outputs/runs') / row['run_name'] / 'run_summary.json'
    payload = json.loads(summary_path.read_text(encoding='utf-8'))
    heatmap = payload.get('depth_attention_rows', [])
    if not heatmap:
        continue
    plt.figure(figsize=(8, 4))
    sns.heatmap(heatmap, cmap='viridis')
    plt.title(f"AttnRes depth attention: {row['run_name']}")
    plt.xlabel('Source index')
    plt.ylabel('Depth-mixing row')
    plt.tight_layout()
    output_path = plot_dir / f"depth_heatmap_{row['run_name']}.png"
    plt.savefig(output_path, dpi=200)
    plt.show()
